#🟦 01-SETUP

In [0]:
from pyspark.sql.functions import (
    col, avg, count, min, max, sum as spark_sum,
    date_format, trunc
)

catalog = "workspace"
schema = "default"
volume = "raw_files"

SILVER_BASE = f"/Volumes/{catalog}/{schema}/{volume}/fuel_dashboard/silver"
GOLD_BASE = f"/Volumes/{catalog}/{schema}/{volume}/fuel_dashboard/gold"

In [0]:
df_prices = spark.read.format("delta").load(f"{SILVER_BASE}/anp_prices")
df_brent = spark.read.format("delta").load(f"{SILVER_BASE}/brent")
df_refinery = spark.read.format("delta").load(f"{SILVER_BASE}/anp_refinery")
df_imports = spark.read.format("delta").load(f"{SILVER_BASE}/anp_imports")

#🟦 GOLD ANP PRICES

#1. Price avg by state/day/product

In [0]:
df_prices_state_day = df_prices.groupBy(
    "data_da_coleta", "estado_sigla", "produto"
).agg(
    avg("valor_de_venda").alias("avg_preco_venda"),
    avg("valor_de_compra").alias("avg_preco_compra"),
    count("*").alias("n_registros")
)

df_prices_state_day.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.prices_state_day"
)

##2. Brazil Price avg 

In [0]:
df_prices_national = df_prices.groupBy(
    "data_da_coleta", "produto"
).agg(
    avg("valor_de_venda").alias("avg_preco_venda"),
    count("*").alias("n_registros")
)

df_prices_national.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.prices_national_day"
)

##3. Price avg by city

In [0]:
df_prices_city = df_prices.groupBy(
    "data_da_coleta", "estado_sigla", "municipio", "produto"
).agg(
    avg("valor_de_venda").alias("avg_preco_venda"),
    count("*").alias("n_registros")
)

df_prices_city.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.prices_city_day"
)

#🟩 GOLD  BRENT

In [0]:
df_brent.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.brent_daily"
)

##5. Brent per month

In [0]:
df_brent_monthly = df_brent.withColumn(
    "year_month", date_format("date", "yyyy-MM")
).groupBy("year_month").agg(
    avg("price").alias("avg_price"),
    min("price").alias("min_price"),
    max("price").alias("max_price")
).orderBy("year_month")

df_brent_monthly.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.brent_monthly"
)

In [0]:
display(df_brent_monthly)

#🟧 GOLD – REFINING (capacity and production)

In [0]:
df_refinery_prod = df_refinery.groupBy(
    "ano", "mes", "refinaria", "produto"
).agg(
    spark_sum("producao").alias("total_producao")
)

df_refinery_prod.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.refinery_production"
)

In [0]:
df_refinery_total = df_refinery.groupBy(
    "ano", "mes", "produto"
).agg(
    spark_sum("producao").alias("total_producao")
)

df_refinery_total.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.refinery_total"
)

#🟥 GOLD – IMPORTS / DEPENDENCY

In [0]:
df_imports_total = df_imports.groupBy(
    "ano", "mes", "produto"
).agg(
    spark_sum("dispendio_receita").alias("total_valor")
)

df_imports_total.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.imports_total"
)


In [0]:
df_gap = df_refinery_total.alias("prod").join(
    df_imports_total.alias("imp"),
    on=["ano", "mes", "produto"],
    how="outer"
).select(
    col("ano"),
    col("mes"),
    col("produto"),
    col("prod.total_producao"),
    col("imp.total_valor"),
    (col("imp.total_valor") - col("prod.total_producao")).alias("gap_estimado")
)
df_gap.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.refining_gap"
)


In [0]:
df_imports_total.select("produto").distinct().show()

In [0]:
df_gap_scenario = df_gap.withColumn(
    "refinarias_100km3",
    (col("gap_estimado") / 100000).cast("double")
).withColumn(
    "refinarias_250km3",
    (col("gap_estimado") / 250000).cast("double")
)

df_gap_scenario.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.refining_scenarios"
)

# Validate


In [0]:
gold_tables = [
    "prices_state_day",
    "prices_national_day",
    "prices_city_day",
    "brent_daily",
    "brent_monthly",
    "refinery_production",
    "refinery_total",
    "imports_total",
    "refining_gap",
    "refining_scenarios"
]

for table in gold_tables:
    df = spark.read.format("delta").load(f"{GOLD_BASE}/{table}")
    print(f"\n{table}")
    print("rows:", df.count())
    display(df.limit(5))

In [0]:
display(df_gap.filter(col("gap_estimado").isNotNull()).orderBy("ano", "mes"))

In [0]:
display(df_gap_scenario.filter(col("gap_estimado").isNotNull()).orderBy("ano", "mes"))